# Postcode Location Validation

This notebook validates Foursquare POI locations by comparing their lat/lng coordinates against the official postcode locations from postcodes.io.

We identify cases where there is an extreme disagreement (>1km) between the POI location and the postcode centroid.

In [ ]:
import geopandas as gpd
import pandas as pd
import polars as pl
import requests
import time
from math import radians, cos, sin, asin, sqrt
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("../src")  # Add path to where 'retail' package lives
import retail.retailfilter_new as rfn

## 1. Load and Filter Foursquare Data

In [ ]:
# Load the raw Foursquare data
data_proc = pl.read_parquet("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
print(f"Raw data: {len(data_proc):,} rows")

In [ ]:
# Apply retail filters
data_proc = rfn.filter_foursquare(data_proc)
print(f"Filtered data: {len(data_proc):,} rows")

In [ ]:
# Convert to pandas for easier processing
data_proc = data_proc.to_pandas()

# Filter to records with valid postcodes
data_with_postcodes = data_proc[data_proc['postcode'].notna() & (data_proc['postcode'] != '')].copy()
print(f"Records with postcodes: {len(data_with_postcodes):,} ({len(data_with_postcodes)/len(data_proc)*100:.1f}%)")

## 2. Haversine Distance Function

Calculate the great circle distance between two points on Earth in kilometers.

In [ ]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    Returns distance in kilometers
    """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # Radius of earth in kilometers
    return c * r

## 3. Fetch Postcode Locations from postcodes.io

We'll use the bulk lookup endpoint to efficiently query multiple postcodes.

In [ ]:
def fetch_postcode_locations_bulk(postcodes, batch_size=100):
    """
    Fetch lat/lng for postcodes using postcodes.io bulk API
    API limit is 100 postcodes per request
    """
    results = {}
    
    # Process in batches
    for i in range(0, len(postcodes), batch_size):
        batch = postcodes[i:i+batch_size]
        
        try:
            # Prepare payload
            payload = {"postcodes": batch}
            
            # Make request
            response = requests.post(
                "https://api.postcodes.io/postcodes",
                json=payload,
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                
                # Extract results
                for item in data.get('result', []):
                    if item['result'] is not None:
                        postcode = item['query']
                        result = item['result']
                        results[postcode] = {
                            'latitude': result.get('latitude'),
                            'longitude': result.get('longitude')
                        }
                    else:
                        # Postcode not found
                        results[item['query']] = {'latitude': None, 'longitude': None}
            
            # Rate limiting - be respectful to the API
            time.sleep(0.1)
            
            # Progress update
            if (i + batch_size) % 1000 == 0:
                print(f"Processed {min(i + batch_size, len(postcodes)):,} / {len(postcodes):,} postcodes")
                
        except Exception as e:
            print(f"Error processing batch {i}-{i+batch_size}: {e}")
            # Continue with next batch
            
    return results

In [ ]:
# Get unique postcodes to minimize API calls
unique_postcodes = data_with_postcodes['postcode'].unique().tolist()
print(f"Unique postcodes to validate: {len(unique_postcodes):,}")

In [ ]:
# Fetch postcode locations from API
# WARNING: This may take several minutes depending on the number of postcodes
print("Fetching postcode locations from postcodes.io...")
postcode_locations = fetch_postcode_locations_bulk(unique_postcodes)
print(f"Retrieved locations for {len(postcode_locations):,} postcodes")

## 4. Calculate Distances and Find Discrepancies

In [ ]:
# Add postcode lat/lng to our data
data_with_postcodes['postcode_lat'] = data_with_postcodes['postcode'].map(
    lambda x: postcode_locations.get(x, {}).get('latitude')
)
data_with_postcodes['postcode_lng'] = data_with_postcodes['postcode'].map(
    lambda x: postcode_locations.get(x, {}).get('longitude')
)

# Filter to records where we successfully got postcode coordinates
data_with_both = data_with_postcodes[
    data_with_postcodes['postcode_lat'].notna() & 
    data_with_postcodes['postcode_lng'].notna()
].copy()

print(f"Records with both POI and postcode coordinates: {len(data_with_both):,}")

In [ ]:
# Calculate distances
data_with_both['distance_km'] = data_with_both.apply(
    lambda row: haversine_distance(
        row['lng'], row['lat'],
        row['postcode_lng'], row['postcode_lat']
    ),
    axis=1
)

print("Distance statistics (km):")
print(data_with_both['distance_km'].describe())

## 5. Identify Extreme Disagreements (>1km)

In [ ]:
# Find cases with >1km disagreement
extreme_discrepancies = data_with_both[data_with_both['distance_km'] > 1.0].copy()
extreme_discrepancies = extreme_discrepancies.sort_values('distance_km', ascending=False)

print(f"\nExtreme discrepancies (>1km): {len(extreme_discrepancies):,}")
print(f"Percentage of validated records: {len(extreme_discrepancies)/len(data_with_both)*100:.2f}%")

# Display top cases
print("\nTop 20 largest discrepancies:")
extreme_discrepancies[[
    'name', 'address', 'postcode', 'lat', 'lng', 
    'postcode_lat', 'postcode_lng', 'distance_km'
]].head(20)

## 6. Visualizations

In [ ]:
# Histogram of distances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# All distances
axes[0].hist(data_with_both['distance_km'], bins=50, edgecolor='black')
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of POI-Postcode Distances (All)')
axes[0].axvline(1.0, color='red', linestyle='--', label='1km threshold')
axes[0].legend()

# Zoomed in to 0-5km
axes[1].hist(data_with_both[data_with_both['distance_km'] <= 5]['distance_km'], 
             bins=50, edgecolor='black')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of POI-Postcode Distances (0-5km)')
axes[1].axvline(1.0, color='red', linestyle='--', label='1km threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Distance ranges
bins = [0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, float('inf')]
labels = ['<100m', '100m-500m', '500m-1km', '1km-2km', '2km-5km', '5km-10km', '>10km']
data_with_both['distance_range'] = pd.cut(data_with_both['distance_km'], bins=bins, labels=labels)

print("\nDistance range distribution:")
print(data_with_both['distance_range'].value_counts().sort_index())
print("\nPercentages:")
print((data_with_both['distance_range'].value_counts(normalize=True) * 100).sort_index().round(2))

## 7. Export Results

In [ ]:
# Save extreme discrepancies to file
output_file = 'postcode_discrepancies_over1km.parquet'
extreme_discrepancies.to_parquet(output_file)
print(f"Saved {len(extreme_discrepancies):,} extreme discrepancies to {output_file}")

In [ ]:
# Create a GeoDataFrame for mapping
# Create points for both POI location and postcode location
from shapely.geometry import Point, LineString

# Create lines connecting POI to postcode location
extreme_discrepancies['geometry'] = extreme_discrepancies.apply(
    lambda row: LineString([
        Point(row['lng'], row['lat']),
        Point(row['postcode_lng'], row['postcode_lat'])
    ]),
    axis=1
)

gdf_discrepancies = gpd.GeoDataFrame(
    extreme_discrepancies, 
    geometry='geometry',
    crs='EPSG:4326'
)

# Save as GeoParquet
gdf_output = 'postcode_discrepancies_over1km.geo.parquet'
gdf_discrepancies.to_parquet(gdf_output)
print(f"Saved GeoDataFrame to {gdf_output}")

## 8. Sample Cases for Manual Review

In [ ]:
# Display detailed information for top 10 discrepancies
print("\nTop 10 discrepancies - detailed view:\n")
for idx, row in extreme_discrepancies.head(10).iterrows():
    print(f"Distance: {row['distance_km']:.2f} km")
    print(f"  Name: {row['name']}")
    print(f"  Address: {row['address']}")
    print(f"  Postcode: {row['postcode']}")
    print(f"  POI Location: ({row['lat']:.6f}, {row['lng']:.6f})")
    print(f"  Postcode Location: ({row['postcode_lat']:.6f}, {row['postcode_lng']:.6f})")
    print(f"  Category: {row.get('fsq_category_labels', 'N/A')}")
    print()